# Image Caption Enhancer

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
import kagglehub
from PIL import Image
from tqdm.notebook import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

## Скачивание датасета

In [ ]:
path = kagglehub.dataset_download("nikhil7280/coco-image-caption")
all_filepaths = []

for root, dirs, files in os.walk(path):
    for file_name in files:
        if file_name.endswith(".jpg"):
            all_filepaths.append(os.path.join(root, file_name))

all_filepaths = sorted(all_filepaths)

print(f"Всего изображений {len(all_filepaths)}")

## Загрузка заголовков COCO

In [ ]:
def load_real_coco_captions(path):
    captions_by_file = {}
    annotation_paths = []
    for root, dirs, files in os.walk(path):
        for file_name in files:
            if file_name in ["captions_train2014.json", "captions_val2014.json"]:
                annotation_paths.append(os.path.join(root, file_name))

    for annotation_path in annotation_paths:
        with open(annotation_path, "r", encoding = "utf-8") as file:
            data = json.load(file)

        id_to_file = {image["id"]: image["file_name"] for image in data["images"]}

        for annotation in data["annotations"]:
            file_name = id_to_file[annotation["image_id"]]
            if file_name not in captions_by_file:
                captions_by_file[file_name] = []
            captions_by_file[file_name].append(annotation["caption"])

    return captions_by_file


real_captions_by_file = load_real_coco_captions(path)

all_filepaths = [
    file_path for file_path in all_filepaths
    if os.path.basename(file_path) in real_captions_by_file
]

print(f"Количество изображений BLIP: {len(all_filepaths)}")
print(f"Количество изображений с заголовками COCO: {len(real_captions_by_file)}")

## Загрузка модели

In [ ]:
device = "cuda"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
model.to(device)
model.eval()
model = model.half()

## Генерация заголовков

In [ ]:
csv_path = "dataset_df.csv"

columns = [
    "file_path",
    "generated_caption_1",
    "generated_caption_2",
    "generated_caption_3",
    "generated_caption_4",
    "generated_caption_5",
    "real_caption_1",
    "real_caption_2",
    "real_caption_3",
    "real_caption_4",
    "real_caption_5"
]
if os.path.exists(csv_path):
    dataset_df = pd.read_csv(csv_path)
    done_paths = set(dataset_df["file_path"])
else:
    dataset_df = pd.DataFrame(columns = columns)
    dataset_df.to_csv(csv_path, index = False)
    done_paths = set()

paths_to_process = [file_path for file_path in all_filepaths if file_path not in done_paths]

batch_size = 8
save_every_batches = 50
buffer = []

progress_bar = tqdm(range(0, len(paths_to_process), batch_size), desc = "BLIP captions")

for batch_number, batch_start in enumerate(progress_bar, start = 1):
    batch_paths = paths_to_process[batch_start:batch_start + batch_size]
    images = [Image.open(file_path).convert("RGB") for file_path in batch_paths]

    inputs = processor(images = images, return_tensors = "pt")
    inputs = {key: value.to(device, dtype = torch.float16) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, num_beams = 5, num_return_sequences = 5, max_new_tokens = 30)

    generated_captions = processor.batch_decode(outputs, skip_special_tokens = True)
    generated_captions = [caption.strip() for caption in generated_captions]

    for image_index, file_path in enumerate(batch_paths):
        start_index = image_index * 5
        image_generated_captions = generated_captions[start_index:start_index + 5]

        while len(image_generated_captions) < 5:
            image_generated_captions.append("")

        file_name = os.path.basename(file_path)
        real_captions = real_captions_by_file.get(file_name, [])[:5]

        while len(real_captions) < 5:
            real_captions.append("")

        row = {
            "file_path": file_path,
            "generated_caption_1": image_generated_captions[0],
            "generated_caption_2": image_generated_captions[1],
            "generated_caption_3": image_generated_captions[2],
            "generated_caption_4": image_generated_captions[3],
            "generated_caption_5": image_generated_captions[4],
            "real_caption_1": real_captions[0],
            "real_caption_2": real_captions[1],
            "real_caption_3": real_captions[2],
            "real_caption_4": real_captions[3],
            "real_caption_5": real_captions[4]
        }

        buffer.append(row)

    if batch_number % save_every_batches == 0:
        pd.DataFrame(buffer, columns = columns).to_csv(csv_path, mode = "a", header = False, index = False)
        buffer = []

        saved_rows = sum(1 for line in open(csv_path, "r", encoding = "utf-8")) - 1
        progress_bar.set_postfix({"saved_rows": saved_rows})

if len(buffer) > 0:
    pd.DataFrame(buffer, columns = columns).to_csv(csv_path, mode = "a", header = False, index = False)

dataset_df = pd.read_csv(csv_path)

In [ ]:
dataset_df.head()